In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw1-task1.ipynb")

# Homework 1 - Task 1
## eDNA and Conventional Fish Surveys

**Based on:** McElroy et al. (2020). *Calibrating environmental DNA metabarcoding to conventional surveys for measuring fish species richness.* Frontiers in Ecology and Evolution.

In this set of exercises you'll wrangle and explore a dataset comparing **eDNA metabarcoding** and **traditional survey methods** for detecting fish species across 68 global freshwater lakes. 

The dataset has the following key columns:

| Column | Description |
|---|---|
| `author` | First author of the primary study |
| `area_ha` | Lake area (hectares) |
| `dna_richness` | Number of fish species detected by eDNA |
| `trad_richness` | Number of fish species detected by conventional surveys |
| `dna_only` | Number of species found **only** by eDNA |
| `trad_only` | Number of species found **only** by conventional surveys |
| `shared` | Number of species found by **both** methods |
| `union` | Total unique species (`dna_only + trad_only + shared`) |
| `marker_cat` | Whether single or multiple genetic markers were used (single vs. multiple eDNA markers) |
| `gear_cat` | Whether single or multiple conventional survey gear types were used (single vs. multiple) |
| `total_vol_liter` | Total volume of water sampled (litres) |

There is no statistics or ML methods in this notebook. Its goal is to review some core Python data wrangling. Along the way you'll practice core pandas skills: loading data, inspecting structure, filtering, grouping, sorting, and deriving new columns.

---

## 1.

Import `pandas` with the alias `pd` and `numpy` with the alias `np`.

In [2]:
# Import necessary packages
import numpy as np
import pandas as pd

## 2. 

Read in the lakes data and store it in a variable called `df`.

In [ ]:
# Read in  lakes data
df = pd.read_csv('lakes_data.csv')

## 3.

How many rows and columns does `df` have? Store the number of rows in a variable called `rows` and the number of columns in a variable called `cols`.

In [9]:
# Get the number of rows and columns in the data frame
print(df.shape)

# Store the number of rows and columns in the data frame in variables
rows = df.shape[0]
cols = df.shape[1]

(68, 17)


In [10]:
grader.check("q3")

q3 results: All test cases passed!

## 4.

Display the data type of each column using `df.info()`. Which columns are numeric and which are categorical?

In [11]:
# Display the data types of each column in the data frame
print(df.dtypes)

author              object
year_pub             int64
area_ha            float64
dna_richness         int64
trad_richness        int64
dna_only             int64
trad_only            int64
shared               int64
union                int64
marker_count         int64
locus               object
marker_cat          object
gear_cat            object
match_effort        object
field_reps           int64
rep_vol_liter      float64
total_vol_liter    float64
dtype: object


In the data frame provided, `year_pub`, `area_ha`, `dna_richness`, `trad_richness`, `dna_only`, `trad_only`, `shared`, `union`, `marker_count`, `field_reps`, `rep_vol_liter`, and `total_col_liter` are numeric columns. 

On the other hand, `author`, `locus`, `marker_cat`, `gear_cat`, and `match_effort` are categorical columns.  

## 5.

Compute summary statistics — mean, median, min, max, and std — for `dna_richness` and `trad_richness`. Store the result in a data frame called `stats` with the following row names `mean`, `median`, `min`, `max`, `std` and column names `dna_richness` and `trad_richness`.



In [ ]:
# Calculate summary statistics for `dna_richness` and `trad_richness`
stats = df[['dna_richness', 'trad_richness']].agg({
    
    'dna_richness': ['mean', 'median', 'min', 'max', 'std'],
    'trad_richness':['mean','median', 'min', 'max', 'std']
}
)

print(stats)

        dna_richness  trad_richness
mean       12.382353      12.073529
median      9.500000      11.000000
min         0.000000       1.000000
max        92.000000      62.000000
std        12.996015       9.692767


In [17]:
grader.check("q5")

q5 results: All test cases passed!

## 6.

How many lakes had eDNA detect **more** species than conventional surveys? How many had conventional surveys detect **more** species? How many were **tied**?

Store your answers in variables called `edna_greater`, `trad_greater`, and `tied`.

In [21]:
# Calculate the number of lakes where `edna_richness` is greater than `trad_richness`
edna_greater = (df['dna_richness'] > df['trad_richness']).sum()

# Calculate the number of lakes where `trad_richness` is greater than `dna_richness`
trad_greater = (df['trad_richness'] > df['dna_richness']).sum()

# Calculate the number of lakes where `dna_richness` and `trad_richness` are tied
tied         = (df['dna_richness'] == df['trad_richness']).sum()

In [22]:
grader.check("q6")

q6 results: All test cases passed!

## 7.

Which lake has the **highest total species richness** (`union`)? Store the row as a `pandas.Series` called `top_lake`, selecting only the `author`, `area_ha`, and `union` columns.

In [23]:
# Identify which lake has the highest total species richness
top_lake = pd.Series(df[['author', 'area_ha', 'union']].loc[df['union'].idxmax()])
top_lake 

author         Doble
area_ha    3290000.0
union            103
Name: 1, dtype: object

Lake 1 has the highest total species richness with a value of 103. 

In [24]:
grader.check("q7")

q7 results: All test cases passed!

## 8.

Create a new column in `df` called `dna_advantage` defined as:

```
dna_advantage = dna_richness - trad_richness
```

A positive value means eDNA found more species; a negative value means conventional surveys found more. Display the first 5 rows of `author`, `dna_richness`, `trad_richness`, and `dna_advantage`.

In [ ]:
# Create a new column called `dna_advantage` that is the difference between `dna_richness` and `trad_richness`
df['dna_advantage'] = df['dna_richness'] - df['trad_richness']

# Display the first 5 rows of the data frame
print(df[['author', 'dna_richness', 'trad_richness', 'dna_advantage']].head())

   author  dna_richness  trad_richness  dna_advantage
0  Civade            14             18             -4
1   Doble            92             62             30
2   Evans            15             10              5
3   Fujii             0              7             -7
4   Fujii             2              8             -6


In [28]:
grader.check("q8")

q8 results: All test cases passed!

## 9.

Filter the dataframe to only lakes where **both** conditions are true:
- `area_ha` is greater than 1000 hectares
- `total_vol_liter` is greater than the median `total_vol_liter`

Store the filtered dataframe in a variable called `large_lakes`. Then store the count of matching lakes in `n_large` and their mean `union` richness in `mean_union_large`.

In [ ]:
# Your answer here. 

# Calculate the number of lakes where area is greater than 1000 ha and total volume is greater than the median total volume (liter)
large_lakes = df[

    # Condition 1: area is greater than 1000 ha
    (df['area_ha']> 1000) & 

    # Condition 2: total volume is greater than the median total volume (liter)
    (df['total_vol_liter'] > df['total_vol_liter'].median())

    ]

# Calculate the number of matching lakes
n_large = len(large_lakes)

# Calculate the mean total species richness for the matching lakes
mean_union_large = large_lakes['union'].mean()

In [31]:
grader.check("q9")

q9 results: All test cases passed!

## 10.

Group lakes by `gear_cat` (single vs. multiple conventional survey methods) **and** `marker_cat` (single vs. multiple eDNA markers). Using `gear_cat` as rows and `marker_cat` as columns, compute a pivot table showing the mean `union` (total species richness) and mean `dna_advantage` for each combination. Round values to 2 decimal places and exclude the `unspecified` category from your interpretation.

Store the result in a variable called `pivot`.

**Hint — `pivot_table` syntax:**
```python
df.pivot_table(
    values=[...],      # columns to aggregate
    index=...,         # variable to use as rows
    columns=...,       # variable to use as columns
    aggfunc="mean"     # aggregation function
).round(2)
```

In [54]:
# Group the data by `gear_cat` and `marker_cat`, and calculate the mean of `union` and `dna_advantage` for each group
pivot = df.pivot_table(values=['union', 'dna_advantage'],
                    index='gear_cat',
                    columns='marker_cat',
                    aggfunc='mean').round(2)

pivot

dna_advantage           union       
marker_cat       multiple single multiple single
gear_cat                                        
multiple             2.67  -2.36    14.33  16.03
single              30.00  -0.60   103.00   8.00
unspecified           NaN   0.00      NaN   1.00

In [55]:
grader.check("q10")

q10 results: All test cases passed!

<!-- BEGIN QUESTION -->

Which combination of `gear_cat` and `marker_cat` detects the most species overall (highest mean `union`)? Answer in the markdown cell below

The combination of `marker_cat = multiple` and `gear_cat = single` has the highest mean species richness (`union = 103`), which is substantially higher than all other combinations. 

<!-- END QUESTION -->



---

**Run the cell below to receive credit for all auto graded questions.**

In [56]:
grader.check_all()

q10 results: All test cases passed!

q3 results: All test cases passed!

q5 results: All test cases passed!

q6 results: All test cases passed!

q7 results: All test cases passed!

q8 results: All test cases passed!

q9 results: All test cases passed!